# CAPA Dashboard - Synthetic Data Generation

Builds a synthetic CAPA dataset on top of cleaned FDA infusion pump recall records.

**Input:** `data/recall_df_clean.csv` — 529 cleaned recall records

**Output:** `data/capa_medfluss.csv` — 529 synthetic CAPA records for MedFluss GmbH

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import random
from datetime import timedelta

In [3]:
DATA_DIR = Path('data')

In [18]:
# Load Data
recall_df = pd.read_csv(DATA_DIR / 'recall_df_clean.csv')
recall_df.shape

(529, 11)

## Define CAPA Dimensions

In [6]:
departments = ['Quality Assurance', 'Regulatory Affairs', 'Clinical Affairs',
                 'Manufacturing', 'Design & Development', 'Supplier Quality']

In [7]:
owners = {
      'Quality Assurance': ['Sarah Müller', 'Thomas Becker'],
      'Regulatory Affairs': ['Priya Nair', 'Klaus Weber'],
      'Clinical Affairs': ['Anna Schmidt', 'David Hoffmann'],
      'Manufacturing': ['Carlos Rivera', 'Lisa Hartmann'],
      'Design & Development': ['Mohammed Al-Farsi', 'Emma Fischer'],
      'Supplier Quality': ['Raj Patel', 'Nina Scholz']
  }

In [8]:
corrected_root_cause_dept_map = {
    'Device Design':            'Design & Development',
    'Software Design':          'Design & Development',
    'Process control':          'Manufacturing',
    'Assembly Error':           'Manufacturing',
    'Human Error — Production': 'Manufacturing',
    'Labeling Execution Error':  'Manufacturing',
    'Nonconforming Material':   'Supplier Quality',
    'Release Without Testing':  'Quality Assurance',
    'Under Investigation':      'Quality Assurance',
    'Labeling Design Error':    'Regulatory Affairs',
    'Documentation Error':      'Regulatory Affairs',
    'Translation Error':        'Regulatory Affairs',
}

recall_df['department'] = recall_df['corrected_root_cause'].map(corrected_root_cause_dept_map)

print(recall_df['department'].value_counts())
print(recall_df['department'].isnull().sum())

department
Design & Development    102
Manufacturing            98
Name: count, dtype: int64
329


In [9]:
def synthetic_open_date():
    start = pd.Timestamp('2022-01-01')
    end = pd.Timestamp('2025-12-31')
    return start + timedelta(days=random.randint(0, (end - start).days))

def synthetic_open_date_open():
    start = pd.Timestamp('2025-08-01')
    end = pd.Timestamp('2026-04-30')
    return start + timedelta(days=random.randint(0, (end - start).days))


In [10]:
severity_weights = {
    'Device Design':            {'Critical': 0.08, 'Major': 0.67, 'Minor': 0.25},
    'Software Design':          {'Critical': 0.08, 'Major': 0.67, 'Minor': 0.25},
    'Process control':          {'Critical': 0.06, 'Major': 0.64, 'Minor': 0.30},
    'Assembly Error':           {'Critical': 0.05, 'Major': 0.60, 'Minor': 0.35},
    'Human Error — Production': {'Critical': 0.04, 'Major': 0.56, 'Minor': 0.40},
    'Labeling Execution Error': {'Critical': 0.03, 'Major': 0.52, 'Minor': 0.45},
    'Nonconforming Material':   {'Critical': 0.08, 'Major': 0.64, 'Minor': 0.28},
    'Release Without Testing':  {'Critical': 0.06, 'Major': 0.59, 'Minor': 0.35},
    'Under Investigation':      {'Critical': 0.04, 'Major': 0.51, 'Minor': 0.45},
    'Labeling Design Error':    {'Critical': 0.03, 'Major': 0.57, 'Minor': 0.40},
    'Documentation Error':      {'Critical': 0.02, 'Major': 0.48, 'Minor': 0.50},
    'Translation Error':        {'Critical': 0.02, 'Major': 0.48, 'Minor': 0.50},
}

In [11]:
def assign_severity(root_cause):
      weights = severity_weights.get(root_cause, {'Critical': 0.2, 'Major': 0.4, 'Minor': 0.4})
      return random.choices(
          list(weights.keys()),
          weights=list(weights.values())
      )[0]

def assign_effectiveness(status):
      if status == 'Open':
          return None
      return random.choices(['Effective', 'Partially Effective', 'Not Effective'],
                             weights=[0.7, 0.2, 0.1])[0]

In [19]:
# Generate CAPA Records
capa_records = []

for i, row in recall_df.iterrows():
    capa_id = f'CAPA-MF-{1000 + len(capa_records):04d}'

    root_cause = row['corrected_root_cause']
    department = corrected_root_cause_dept_map.get(root_cause, 'Quality Assurance')
    owner = random.choice(owners[department])

    if row['recall_status'] in ['Terminated', 'Completed']:
        open_date = synthetic_open_date()
    else:
        open_date = synthetic_open_date_open()

    severity = assign_severity(root_cause)

    if row['recall_status'] in ['Terminated', 'Completed']:
        capa_status = 'Closed'
        max_days = {'Critical': 60, 'Major': 120, 'Minor': 240}[severity]
        close_date = open_date + timedelta(days=random.randint(14, max_days))
    else:
        capa_status = 'Open'
        close_date = None

    if capa_status == 'Closed' and pd.notna(close_date) and pd.notna(open_date):
        days_open = (close_date - open_date).days
    elif capa_status == 'Open' and pd.notna(open_date):
        days_open = (pd.Timestamp.today() - open_date).days
    else:
        days_open = None

    overdue_threshold = {'Critical': 30, 'Major': 90, 'Minor': 180}
    threshold = overdue_threshold[severity]
    overdue = 'Yes' if (capa_status == 'Open' and days_open and days_open > threshold) else 'No'

    effectiveness = assign_effectiveness(capa_status)

    capa_records.append({
        'capa_id': capa_id,
        'company': 'MedFluss GmbH',
        'recall_id': row['cfres_id'],
        'product_res_number': row['product_res_number'],
        'recalling_firm': row['recalling_firm'],
        'product_description': row['product_description'],
        'nonconformity': row['reason_for_recall'],
        'original_root_cause': row['root_cause_description'],
        'corrected_root_cause': root_cause,
        'department': department,
        'assigned_owner': owner,
        'severity': severity,
        'capa_status': capa_status,
        'open_date': open_date,
        'close_date': close_date,
        'days_open': days_open,
        'overdue': overdue,
        'corrective_action': row['action'],
        'effectiveness_result': effectiveness
    })

capa_df = pd.DataFrame(capa_records)

In [20]:
# Validate
print(capa_df.shape)
capa_df.head()

(529, 19)


,capa_id,company,recall_id,product_res_number,recalling_firm,product_description,nonconformity,original_root_cause,corrected_root_cause,department,assigned_owner,severity,capa_status,open_date,close_date,days_open,overdue,corrective_action,effectiveness_result
0,CAPA-MF-1000,MedFluss GmbH,85112,Z-0147-2010,Hospira Inc,Power cord for QVue Continuous Cardiac Output ...,Fire/Shock hazard-- The power cord used in the...,Component design/selection,Component design/selection,Quality Assurance,Thomas Becker,Major,Closed,2022-08-21,2022-11-22,93,No,Hospira initiated its recall on 08/14/2009. A...,Effective
1,CAPA-MF-1001,MedFluss GmbH,105064,Z-0268-2012,Wolf Medical Supply Inc.,"Wolf Medical Supply, Inc., WOLF-PAK REDI-FLO ...",Redi-Flo Elastomeric Infusion Pumps may have a...,Process control,Process control,Manufacturing,Lisa Hartmann,Major,Closed,2022-07-10,2022-10-30,112,No,"On 10/20/2011 Wolf Medical Supply Inc., custom...",Partially Effective
2,CAPA-MF-1002,MedFluss GmbH,105401,Z-0269-2012,Wolf Medical Supply Inc.,"Wolf Medical Supply Inc., WOLF-PAK REDI-FLO ...",Redi-Flo Elastomeric Infusion Pumps may have a...,Process control,Process control,Manufacturing,Carlos Rivera,Major,Closed,2023-09-01,2023-09-22,21,No,"On 10/20/2011 Wolf Medical Supply Inc., custom...",Effective
3,CAPA-MF-1003,MedFluss GmbH,104072,Z-3284-2011,Hospira Inc.,Plum A+ Single Channel Infusion Pumps; Hospira...,Hospira has received reports of incorrect seat...,Nonconforming Material/Component,Nonconforming Material/Component,Quality Assurance,Thomas Becker,Major,Closed,2025-11-10,2026-01-16,67,No,"Hospira, Inc. sent an ""URGENT DEVICE RECALL"" l...",Effective
4,CAPA-MF-1004,MedFluss GmbH,107986,Z-1338-2012,Medtronic Neuromodulation,"Medtronic, Model 8870, Application Software Ca...",Medtronic has confirmed that an algorithm used...,Software design,Software design,Quality Assurance,Thomas Becker,Critical,Closed,2022-08-28,2022-10-12,45,No,"Medtronic mailed an ""Urgent Medical Device Cor...",Effective


In [15]:
print(capa_df.shape)
print(capa_df['severity'].value_counts())
print(capa_df['department'].value_counts())
print(capa_df['overdue'].value_counts())

(529, 19)
severity
Major       279
Minor       185
Critical     65
Name: count, dtype: int64
department
Quality Assurance       329
Design & Development    102
Manufacturing            98
Name: count, dtype: int64
overdue
No     468
Yes     61
Name: count, dtype: int64


In [21]:
# Save
capa_df.to_csv(DATA_DIR / 'capa_medfluss.csv', index=False)
print("Saved successfully")

Saved successfully
